In [1]:
import sys
import os

%load_ext autoreload
%autoreload 2

repo_dir = os.path.dirname(os.path.abspath("./"))
if repo_dir not in sys.path:
    print(f'add repository dir: {repo_dir}')
    sys.path.append(repo_dir)

for p in list(sys.path):
    if 'recurrent-memory-transformer' in p:
        sys.path.remove(p)

from transformers import AutoModel, AutoTokenizer
from rl.bert_predictor import BertPredictor
from rl.text_env import TextEnv, TextReplayBuffer
from rl import WordsCounterEnv
from rl.bert_predictor import BertPredictor
from datasets import load_dataset
from rl.sacd import SAC, SACArgs


bert_name = "haisongzhang/roberta-tiny-cased"
tokenizer = AutoTokenizer.from_pretrained(bert_name, use_fast=True, revision="main")
bert_model = AutoModel.from_pretrained(bert_name, revision="main")
predictor = BertPredictor(bert_model, 4, tokenizer, 512, 256, 1).cuda()

add repository dir: /home/nazar/projects/multi-step-retrieval-rl


/home/nazar/torchenv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
os.environ['TOKENIZERS_PARALLELISM'] = 'true'
print("loading dataset: AIRI-NLP/quality_counter_new_1024")
dataset = load_dataset("AIRI-NLP/quality_counter_new_1024")

loading dataset: AIRI-NLP/quality_counter_new_1024


In [7]:
action_embed = BertPredictor(bert_model, 4, tokenizer, 512, 256, 1).cuda()
action_embed_target = BertPredictor(bert_model, 4, tokenizer, 512, 256, 1).cuda()

state_embed = BertPredictor(bert_model, 4, tokenizer, 512, 256, 1).cuda()
state_embed_target = BertPredictor(bert_model, 4, tokenizer, 512, 256, 1).cuda()

agent = SAC(
    state_embed, action_embed, state_embed_target, action_embed_target, 3, 
    SACArgs(gamma=0.99, alpha=0.2, tau=0.005, target_update_interval=1, automatic_entropy_tuning=True, lr=1e-4)
)

env = WordsCounterEnv(
    dataset, 
    block_size=32, 
    embedder=agent.action_embed_target, 
    max_length=1024, 
    embed_tokenizer=tokenizer, 
    max_steps_count=6,
    add_question=True
)

buffer = TextReplayBuffer(100_000, tokenizer = tokenizer)

In [8]:
from tqdm import tqdm
import torch

learning_start = 2_000

step = 0
R = 0
for ep_number in tqdm(range(100_000)):

    s = env.reset()
    done = False
    a_embeds = env.get_extra_embeds(agent.critic.action_embed)
    agent.policy.update(agent.critic)

    qf_loss, alpha_loss = 0, 0

    a_all = []

    while not done:
        step += 1
        
        action, logp, entropy = agent.select_action(s, a_embeds, random = step < learning_start)
        s_next, a_data, reward, done = env.step(action)
        buffer.add(s, a_data, s_next, reward, done, entropy)
        s = s_next
        R += reward
        a_all.append(action)
        
        if step > learning_start and step % 5 == 0:
            s_batch, a_batch, next_s_batch, r_batch, not_done_batch, entropy_batch = buffer.sample(64)
            qf_loss, alpha_loss = agent.update(s_batch, a_batch, next_s_batch, r_batch, not_done_batch, entropy_batch, step)
    
    if ep_number % 100 == 0 and ep_number > 0:
        print(R / 100, qf_loss, alpha_loss, agent.alpha)
        print(a_all)
        R = 0


  0%|          | 0/100000 [00:00<?, ?it/s]

  0%|          | 109/100000 [00:02<39:53, 41.74it/s]

0.3 0 0 0.2
[28, 13, 14, 31, 26, 19]


  0%|          | 209/100000 [00:05<40:46, 40.80it/s]

0.24 0 0 0.2
[5, 11, 9, 21, 5, 30]


...

0.39 0.18168184161186218 -6.621524333953857 0.9945917725563049
[22, 28, 31, 26, 9, 18]


  3%|▎         | 2802/100000 [07:48<4:57:13,  5.45it/s]

0.29 0.14056076109409332 -6.977119445800781 1.0416215658187866
[6, 8, 16, 11, 14, 15]


  3%|▎         | 2898/100000 [08:07<4:31:59,  5.95it/s]


KeyboardInterrupt: 